# Campanha de pesquisa no Colab

Roda várias fatias (`--shard I/N`) da grade do `run_research.py` **ao mesmo tempo**, com um
único Ollama na GPU atendendo as requisições em paralelo. Cada fatia contém blocos pareados
completos e grava em uma pasta própria no Google Drive. No fim, `analysis/analyze_completion.py`
recebe as pastas de todas as fatias, inclusive as da máquina local.

Regras para a consolidação ser aceita:

- Todas as máquinas usam o **mesmo commit**, o mesmo modelo (digest conferido abaixo),
  o mesmo `SEED_COUNT` e o mesmo `N_SHARDS`.
- Cada fatia roda em **uma só máquina**. A local fica com as fatias que não estiverem em `SHARDS`.
- Execute as fatias em ordem crescente; parar por tempo é aceitável, escolher fatias pelo resultado não.
- Ambiente com GPU (Runtime > Change runtime type). L4 ou A100 comportam mais fatias em paralelo que a T4.
- Repositório privado: crie em Secrets (ícone de chave) o segredo `GITHUB_TOKEN`, com um token de leitura do repositório.
  A célula do clone precisa ser executada pela interface do Colab: segredos não são lidos por execução remota.
- Sem execução em segundo plano no plano Pro, mantenha a aba aberta.

In [ ]:
# Ajuste COMMIT e SHARDS; SEED_COUNT e N_SHARDS devem ser iguais aos da máquina local.
COMMIT = ""  # saída de `git rev-parse HEAD` na máquina local
SEED_COUNT = 24  # 4.224 casos; igual ao padrão do run_research.py
N_SHARDS = 8  # três sementes e 528 casos por fatia; a máquina local fica com a fatia 0
SHARDS = [1, 2, 3]  # fatias rodadas aqui, em paralelo
HOURS = 20  # orçamento desta sessão; renova a cada retomada
BASE_DIR = "/content/drive/MyDrive/brazilianinfosim/conclusao"

MODEL = "qwen3:4b-instruct-2507-q4_K_M"
EXPECTED_DIGEST = "0edcdef34593eac1aa2be9c7d06c432dcf81945adca5eca2f27662c18f168ba0"
OLLAMA_VERSION = "0.20.5"
PYTHON_VERSION = "3.14.3"
REPO = "yrlanrteixeira/brazilianinfosim"
assert COMMIT, "Preencha COMMIT com o mesmo commit da máquina local."
assert SEED_COUNT % N_SHARDS == 0 and all(0 <= s < N_SHARDS for s in SHARDS)

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import base64, subprocess, urllib.error, urllib.request
token = userdata.get("GITHUB_TOKEN").strip()
# Token no cabeçalho, fora da URL, para não aparecer em mensagens de erro.
request = urllib.request.Request(f"https://api.github.com/repos/{REPO}", headers={"Authorization": f"Bearer {token}"})
try:
    urllib.request.urlopen(request)
except urllib.error.HTTPError as error:
    raise RuntimeError({401: "Token inválido ou expirado.",
                        404: "Token sem acesso ao repositório: confira o dono e o repositório selecionado."}.get(error.code, error.code))
header = "Authorization: Basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
subprocess.run(["rm", "-rf", "/content/brazilianinfosim"], check=True)
for command in (["git", "-c", f"http.extraHeader={header}", "clone", "-q", f"https://github.com/{REPO}.git", "/content/brazilianinfosim"],
                ["git", "-C", "/content/brazilianinfosim", "checkout", "-q", COMMIT]):
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr.replace(token, "***"))
!git -C /content/brazilianinfosim log --oneline -1


O ambiente Python fica isolado do Colab, com a mesma versão do Python e os pacotes fixados
em `requirements.txt`. O PyTorch de CPU basta: a campanha não calcula embeddings.

In [ ]:
%cd /content/brazilianinfosim
!pip install -q uv
!uv python install {PYTHON_VERSION}
!uv venv -q --clear --python {PYTHON_VERSION} /content/venv
!uv pip install -q --python /content/venv/bin/python -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cpu --index-strategy unsafe-best-match
!/content/venv/bin/python --version

In [ ]:
import json, os, time, urllib.request
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh > /dev/null
# Uma vaga de inferência por fatia; o modelo fica carregado durante toda a sessão.
env = {**os.environ, "OLLAMA_NUM_PARALLEL": str(len(SHARDS)), "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_KEEP_ALIVE": "-1"}
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull {MODEL}
tags = json.load(urllib.request.urlopen("http://localhost:11434/api/tags"))["models"]
version = json.load(urllib.request.urlopen("http://localhost:11434/api/version"))["version"]
digest = next(m["digest"] for m in tags if m["name"] == MODEL)
print(version, digest)
assert digest == EXPECTED_DIGEST, "Modelo diferente do local; as fatias não poderão ser juntadas."
assert version == OLLAMA_VERSION, "Versão do Ollama diferente da combinada."

Inicia uma execução por fatia e acompanha o progresso. Se a sessão cair, rode de novo todas
as células: casos completos não se repetem e falhas ou interrupções são retentadas.

In [ ]:
from pathlib import Path
os.makedirs(f"{BASE_DIR}/logs", exist_ok=True)
# O Colab exporta um backend inline do matplotlib que o Python isolado não aceita.
worker_env = {**os.environ, "MPLBACKEND": "Agg"}
workers = {}
for shard in SHARDS:
    folder = Path(BASE_DIR, f"shard-{shard}-de-{N_SHARDS}")
    command = ["/content/venv/bin/python", "scripts/run_research.py", "--output", str(folder),
               "--seed-count", str(SEED_COUNT), "--shard", f"{shard}/{N_SHARDS}", "--hours", str(HOURS)]
    if (folder / "campaign.json").exists():
        command += ["--resume", "--retry-incomplete"]
    log = open(f"{BASE_DIR}/logs/shard-{shard}-de-{N_SHARDS}.log", "a")
    workers[shard] = (folder, subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT,
                                               cwd="/content/brazilianinfosim", env=worker_env))


In [ ]:
def report():
    for shard, (folder, process) in workers.items():
        status_path = folder / "status.json"
        status = json.loads(status_path.read_text(encoding="utf-8")) if status_path.exists() else {}
        state = "rodando" if process.poll() is None else f"saiu com {process.returncode}"
        print(f"fatia {shard}: {status.get('success', 0)}/{status.get('planned', '?')} sucesso, "
              f"{status.get('failed', 0)} falhas, {status.get('complete_blocks', 0)} blocos, {state}", flush=True)

report()


In [ ]:
# Acompanhamento contínuo; interromper esta célula não para as fatias.
while any(process.poll() is None for _, process in workers.values()):
    time.sleep(600)
    print(time.strftime("%H:%M"), flush=True)
    report()


In [ ]:
# Diagnóstico de uma fatia que saiu com erro.
!tail -n 30 "{BASE_DIR}/logs/shard-{SHARDS[0]}-de-{N_SHARDS}.log"